In [2]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit

# Step 1: Load HC3 raw dataset
raw = load_dataset("json", data_files="https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/all.jsonl")["train"]
raw_df = raw.to_pandas()
raw_df["question_id"] = raw_df.index

# Flatten nested lists of human and chatgpt answers while retaining question_id
texts, labels, qids = [], [], []
for _, row in raw_df.iterrows():
    for h in row["human_answers"]:
        if h.strip():
            texts.append(h)
            labels.append(0)  # 0 = human
            qids.append(row["question_id"])
    for a in row["chatgpt_answers"]:
        if a.strip():
            texts.append(a)
            labels.append(1)  # 1 = AI-generated
            qids.append(row["question_id"])

df = pd.DataFrame({"text": texts, "label": labels, "question_id": qids})
print("Label Distribution:")
print(df["label"].value_counts(normalize=True))

# Check word count distribution to confirm 256 sequence truncation max_length
word_counts = df["text"].str.split().str.len()
print("\nWord Count Stats:")
print(word_counts.describe(percentiles=[0.5, 0.75, 0.90, 0.95]))

# Step 2: Leakage-Free Grouped Split on question_id (80/20)
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df["question_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print(f"\nGrouped Split -> Train: {len(train_df)} | Val: {len(val_df)}")

Label Distribution:
label
0    0.685302
1    0.314698
Name: proportion, dtype: float64

Word Count Stats:
count    85431.000000
mean       146.234049
std        141.134892
min          1.000000
50%        118.000000
75%        193.000000
90%        269.000000
95%        355.000000
max       7904.000000
Name: text, dtype: float64

Grouped Split -> Train: 68314 | Val: 17117


In [3]:
from datasets import Dataset
from transformers import AutoTokenizer

# Step 3: Initialize DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

# Convert DataFrames to Hugging Face Datasets and map tokenization
train_ds = Dataset.from_pandas(train_df).map(tokenize_batch, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize_batch, batched=True)

# Format outputs as PyTorch Tensors
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/68314 [00:00<?, ? examples/s]

Map:   0%|          | 0/17117 [00:00<?, ? examples/s]

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, roc_auc_score
from tqdm.notebook import tqdm  # <-- Added progress bar import

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(device)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# Setup LR Scheduler with 10% Warmup
epochs = 2
num_training_steps = epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(0.1 * num_training_steps), 
    num_training_steps=num_training_steps
)

history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": [], "val_auc": []}

for epoch in range(epochs):
    # --- Training Phase ---
    model.train()
    train_loss = 0.0
    
    # Wrapped train_loader with tqdm
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for batch in train_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        
        optimizer.zero_grad()
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = out.loss
        loss.backward()
        
        optimizer.step()
        scheduler.step()  # Step LR schedule
        
        train_loss += loss.item() * input_ids.size(0)
        
        # Display current loss live inside the progress bar
        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    preds, probs, y_true = [], [], []
    
    # Wrapped val_loader with tqdm
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")
    with torch.no_grad():
        for batch in val_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += out.loss.item() * input_ids.size(0)
            
            # Extract class 1 (AI) probabilities for ROC-AUC
            batch_probs = torch.softmax(out.logits, dim=1)[:, 1].cpu().tolist()
            logits_preds = out.logits.argmax(dim=1).cpu().tolist()
            
            probs.extend(batch_probs)
            preds.extend(logits_preds)
            y_true.extend(labels.cpu().tolist())

    epoch_train_loss = train_loss / len(train_ds)
    epoch_val_loss = val_loss / len(val_ds)
    epoch_val_acc = sum(p == t for p, t in zip(preds, y_true)) / len(y_true)
    epoch_val_f1 = f1_score(y_true, preds)
    epoch_val_auc = roc_auc_score(y_true, probs)

    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)
    history["val_f1"].append(epoch_val_f1)
    history["val_auc"].append(epoch_val_auc)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_train_loss:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f} | "
          f"Val F1: {epoch_val_f1:.4f} | Val AUC: {epoch_val_auc:.4f}")

Training on device: cpu


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/2 [Train]:   0%|          | 0/4270 [00:00<?, ?it/s]

Epoch 1/2 [Val]:   0%|          | 0/535 [00:00<?, ?it/s]

Epoch 1/2 | Train Loss: 0.0516 | Val Loss: 0.0173 | Val Acc: 0.9943 | Val F1: 0.9910 | Val AUC: 0.9999


Epoch 2/2 [Train]:   0%|          | 0/4270 [00:00<?, ?it/s]

Epoch 2/2 [Val]:   0%|          | 0/535 [00:00<?, ?it/s]

Epoch 2/2 | Train Loss: 0.0034 | Val Loss: 0.0191 | Val Acc: 0.9947 | Val F1: 0.9916 | Val AUC: 0.9999


In [5]:
from sklearn.metrics import classification_report

print(f"Final Validation Accuracy: {history['val_acc'][-1]:.4f}")
print(f"Final Validation F1:       {history['val_f1'][-1]:.4f}")
print(f"Final Validation ROC-AUC:  {history['val_auc'][-1]:.4f}\n")
print("Detailed Classification Report:\n")
print(classification_report(y_true, preds, target_names=["human", "ai"]))

Final Validation Accuracy: 0.9947
Final Validation F1:       0.9916
Final Validation ROC-AUC:  0.9999

Detailed Classification Report:

              precision    recall  f1-score   support

       human       1.00      0.99      1.00     11721
          ai       0.99      1.00      0.99      5396

    accuracy                           0.99     17117
   macro avg       0.99      1.00      0.99     17117
weighted avg       0.99      0.99      0.99     17117



In [6]:
import os
import json

os.makedirs("../models/ai_text", exist_ok=True)

# Export fine-tuned weights and tokenizer
model.save_pretrained("../models/ai_text/model_v1")
tokenizer.save_pretrained("../models/ai_text/model_v1")

# Export evaluation metrics and history metadata
with open("../models/ai_text/metrics_v1.json", "w") as f:
    json.dump({
        "accuracy": float(history["val_acc"][-1]),
        "f1": float(history["val_f1"][-1]),
        "roc_auc": float(history["val_auc"][-1]),
        "class_names": ["human", "ai"],
        "label_map": {"0": "human", "1": "ai"},
        "training_history": history,
    }, f, indent=2)

# Extract CLS token reference embeddings (500 validation samples)
model.eval()
embeddings = []
ref_loader = DataLoader(val_ds.select(range(500)), batch_size=32, shuffle=False)

with torch.no_grad():
    for batch in ref_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        out = model.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]  # Extract CLS token vector [B, 768]
        embeddings.append(pooled.cpu())

reference_embeddings = torch.cat(embeddings)[:500]
torch.save(reference_embeddings, "../models/ai_text/reference_embeddings_v1.pt")

print("Successfully exported DistilBERT model, tokenizer, metrics, and CLS reference embeddings!")
print(f"Reference Embeddings Matrix Shape: {reference_embeddings.shape}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Successfully exported DistilBERT model, tokenizer, metrics, and CLS reference embeddings!
Reference Embeddings Matrix Shape: torch.Size([500, 768])
